# Sentiment Analysis

Ensure that FastAPI is up and running. [instructions here](../fast_api/readme.md)

# Table of contents

1. [Data prep](#1-data)

2. [NeXT](#2-next)

3. [Standard](#3-standard)

In [1]:
# imports
import requests
import json

In [2]:
FAST_API_URL = "http://localhost:9000"

 # 1. Data prep <a class="anchor" id="1-data"></a>

We use a subset of [IMDB sentiment analysis](https://ai.stanford.edu/~amaas/data/sentiment/) dataset in this demo.

In [3]:
sa_data = json.load(open('sa_data.json'))

In [4]:
sa_unlabeled_data = sa_data['unlabeled_data']
sa_unlabeled_data[0]

'The idea of making a film about the Beatles sounds doomed idea, as no production can catch the idea of the actual historic Beatles. Then it is perhaps best not to try to recreate the past, but to produce an illustration that works best with the other available Beatles material. This is exactly what \'Birth of the Beatles\' offers to us, the simple story known to us without any extravaganza.<br /><br />*** SPOILERS here on *** <br /><br />Be warned that not everything is that accurate as some Beatles-graduates might expect. The Beatles are seen performing songs that hardly were even composed by that time. The Beatles perform Ask Me Why"'

In [5]:
sa_explanation_triples = sa_data['explanation_triples']
sa_explanation_triples[0]

{'text': 'Walt Disney\'s CINDERELLA takes a story everybody\'s familiar with and embellishes it with humor and suspense, while retaining the tale\'s essential charm. Disney\'s artists provide the film with an appealing storybook look that emanates delectable fairy tale atmosphere. It is beautifully, if conventionally, animated; the highlight being the captivating scene where the Fairy Godmother transforms a pumpkin into a majestic coach and Cinderella\'s rags to a gorgeous gown. Mack David, Al Hoffman, and Jerry Livingston provide lovely songs like A Dream Is a Wish Your Heart Makes" and "Bibbidi-Bobbidi-Boo" that enhance both the scenario and the characters.<br /><br />Even though CINDERELLA\'s story is predictable',
 'label': 'Positive',
 'explanation': "The word 'embellishes' appears in the text"}

# 2. NeXT <a class="anchor" id="2-next"></a>

Check [this](../fast_api/json_schema.py#L293) json schema for a list of all parameters.

Check [this](../model_training/internal_api/defaults.py) for default values.

In [ ]:
sa_next_params = {
    "experiment_name": "imdb_sa_nle",
    "dataset_name": "imdb_sa_nle",
    "dataset_size": len(sa_unlabeled_data),
    "task": "sa",
    "match_batch_size": 50,
    "unlabeled_batch_size": 100,
    "learning_rate": 0.1,
    "epochs": 5,
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    "emb_dim": 100,
    "gamma": 0.5,
    "hidden_dim": 100,
    "random_state": 7698,
    "start_epoch": 0,
    "pre_train_hidden_dim": 300,
    "pre_train_training_size": 50000,
    "soft_match": True,
    "stage": "both",
    "pre_train_build_data": True,
    "build_data": True,
    "load_model": False, # when setting to True ensure other parameters are same, we currently don't check that
}
sa_next_label_space = {
    "Positive": 0,
    "Negative": 1
}

In [ ]:
response = requests.post(
    FAST_API_URL + '/training/next/api/',
    json={
        'params': sa_next_params,
        'label_space': sa_next_label_space,
        'explanation_triples': sa_explanation_triples,
        'unlabeled_text': sa_unlabeled_data,
    }
)
print(response.text)

# 3. Standard <a class="anchor" id="3-standard"></a>

Check [this](../fast_api/json_schema.py#L516) json schema for a list of all parameters.

Check [this](../model_training/internal_api/defaults.py) for default values.

In [ ]:
sa_standard_labeled_data = list(map(
    lambda x: {'text': x['text'], 'label': x['label']}, 
    sa_explanation_triples
))

In [ ]:
sa_standard_params = {
    "experiment_name": "imdb_sa_nle",
    "dataset_name": "imdb_sa_nle",
    "task": "sa",
    "match_batch_size": 50,
    "learning_rate": 0.1,
    "epochs": 5,
    "embeddings": "charngram.100d", # usual default: "glove.840B.300d"
    "emb_dim": 100,
    "gamma": 0.5,
    "hidden_dim": 100,
    "random_state": 7698,
    "start_epoch": 0,
    "stage": "both",
    "build_data": True,
    "load_model": False, # when setting to True ensure other parameters are same, we currently don't check that
}
sa_standard_label_space = {
    "Positive": 0,
    "Negative": 1
}

In [ ]:
response = requests.post(
    FAST_API_URL + '/training/standard/api/',
    json={
        'params': sa_standard_params,
        'label_space': sa_standard_label_space,
        'labeled_data': sa_standard_labeled_data,
    }
)
print(response.text)